# FUNCIONES de Pyspark
- explode: extare un array de una columna
- coalesce: combina varias columnas en una 

In [7]:
import findspark
findspark.init()

In [8]:
from pyspark.sql.functions import col, explode, array, udf, when, coalesce
from pyspark.sql import SparkSession
from pyspark.sql.types import StringType


In [9]:
spark = SparkSession.builder.getOrCreate()

In [10]:
data = [("Alice", ["Badminton", "Tennis"]), ("Bob", ["Tennis", "Criccket"]), ("Julie", ["Cricket", "Carroms"])]

In [11]:
df = spark.createDataFrame(data,["name","hobbies"])

Traceback (most recent call last):
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\serializers.py", line 437, in dumps
    return cloudpickle.dumps(obj, pickle_protocol)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 563, in dump
    return Pickler.dump(self, obj)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 653, in reducer_override
    return self._function_reduce(obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 526, in _function_reduce
    return self._dynamic_function_reduce(obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:

PicklingError: Could not serialize object: IndexError: tuple index out of range

In [7]:
df.show()

+-----+-------------------+
| name|            hobbies|
+-----+-------------------+
|Alice|[Badminton, Tennis]|
|  Bob| [Tennis, Criccket]|
|Julie| [Cricket, Carroms]|
+-----+-------------------+



In [8]:
df_explode = df.select(col("name"), explode(df.hobbies).alias("hobbies_explode"))
df_explode.show()

+-----+---------------+
| name|hobbies_explode|
+-----+---------------+
|Alice|      Badminton|
|Alice|         Tennis|
|  Bob|         Tennis|
|  Bob|       Criccket|
|Julie|        Cricket|
|Julie|        Carroms|
+-----+---------------+



# FUNCIONES 
- coalesce : Combina multiples columnas 

In [12]:
data_info = [("MD","", "ZR"),("", "ZR", None),(None, "", "BRC")]
schema = "City1 string, City2 string, City3 string"
df_city = spark.createDataFrame(data_info, schema)
df_city.show()

Traceback (most recent call last):
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\serializers.py", line 437, in dumps
    return cloudpickle.dumps(obj, pickle_protocol)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 73, in dumps
    cp.dump(obj)
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 563, in dump
    return Pickler.dump(self, obj)
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 653, in reducer_override
    return self._function_reduce(obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\app\opt\spark\spark-3.2.4-bin-hadoop2.7\python\pyspark\cloudpickle\cloudpickle_fast.py", line 526, in _function_reduce
    return self._dynamic_function_reduce(obj)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:

PicklingError: Could not serialize object: IndexError: tuple index out of range

In [12]:
df_city = (df_city.withColumn("City1", when(col("City1")=='', None).otherwise(col("City1")))
           .withColumn("City2", when(col("City2")=='', None).otherwise(col("City2")))
           .withColumn("City3", when(col("City3")=='', None).otherwise(col("City3"))))
df_result = df_city.withColumn("Result", coalesce(col("City1"), col("City2"), col("City3"))).select("Result")
df_result.show()

+------+
|Result|
+------+
|    MD|
|    ZR|
|   BRC|
+------+



In [13]:
@udf(returnType=StringType())
def city_value(columnas_city):
    cities = ""
    for city in columnas_city:
        if city is not None and city !="null":
            cities = city
    return cities

In [14]:
lista_cities = [ x for x in df_city.columns if "City" in x ]
print(lista_cities)

['City1', 'City2', 'City3']


In [15]:
df_result = df_city.withColumn("Result", city_value(array(lista_cities)))
df_result.show()

+-----+-----+-----+------+
|City1|City2|City3|Result|
+-----+-----+-----+------+
|   MD| null|   ZR|    ZR|
| null|   ZR| null|    ZR|
| null| null|  BRC|   BRC|
+-----+-----+-----+------+

